# Tracking Israelite Travel Using Text Mining
Antony Klar ak2433

In [ ]:
# Upgrade the transformers library to ensure 'pipeline' is importable
import pandas as pd
import numpy as np
import kagglehub
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline
from tqdm.notebook import tqdm
import re
import spacy
import plotly.express as px
import plotly.graph_objects as go

First we download the dataset from kaggle. I decided to use the KJV version because it is the one I have been using this semester and there is also an available dataset on it.

In [ ]:
file_path = "t_kjv.csv"

dataset_path = kagglehub.dataset_download(
  "oswinrh/bible"
)
# Construct the full path to the CSV file within the downloaded dataset
full_file_path = f"{dataset_path}/{file_path}"

df = pd.read_csv(full_file_path)

df.head(-5)

100%|██████████| 176M/176M [00:04<00:00, 37.7MB/s]

Extracting files...


,id,b,c,v,t
0,1001001,1,1,1,In the beginning God created the heaven and th...
1,1001002,1,1,2,"And the earth was without form, and void; and ..."
2,1001003,1,1,3,"And God said, Let there be light: and there wa..."
3,1001004,1,1,4,"And God saw the light, that it was good: and G..."
4,1001005,1,1,5,"And God called the light Day, and the darkness..."
...,...,...,...,...,...
31093,66022012,66,22,12,"And, behold, I come quickly; and my reward is ..."
31094,66022013,66,22,13,"I am Alpha and Omega, the beginning and the en..."
31095,66022014,66,22,14,"Blessed are they that do his commandments, tha..."
31096,66022015,66,22,15,"For without are dogs, and sorcerers, and whore..."


Now we have every verse of the Bible represented in a dataframe. Since this is a class on The Old Testament, we are going to drop all the New Testament books for this project.

In [ ]:
df_old_testament = df.iloc[:39]

In [ ]:
OT_df = df[df['b'] <= 39]
display(OT_df.head(-5))

,id,b,c,v,t
0,1001001,1,1,1,In the beginning God created the heaven and th...
1,1001002,1,1,2,"And the earth was without form, and void; and ..."
2,1001003,1,1,3,"And God said, Let there be light: and there wa..."
3,1001004,1,1,4,"And God saw the light, that it was good: and G..."
4,1001005,1,1,5,"And God called the light Day, and the darkness..."
...,...,...,...,...,...
23135,39003015,39,3,15,"And now we call the proud happy; yea, they tha..."
23136,39003016,39,3,16,Then they that feared the LORD spake often one...
23137,39003017,39,3,17,"And they shall be mine, saith the LORD of host..."
23138,39003018,39,3,18,"Then shall ye return, and discern between the ..."


I found a dataset on all the locations mentioned in the Bible and their corresponding modern locations as well as their coordinates. This will come in handy when we extract the locations the Israelites travelled to.

In [ ]:
# Raw URL for the CSV file on GitHub (corrected to raw content)
locations_url = 'https://raw.githubusercontent.com/BradyStephenson/bible-data/main/BibleData-Place.csv'

# Load the CSV data directly into a pandas DataFrame
df_locations = pd.read_csv(locations_url)

# Display the last few rows of the DataFrame (corrected display call)
display(df_locations.tail(10))

,place_id,place_name,place_type,modern_equivalent,place_notes,openbible_id,openbible_url,name_instance,place_sequence
108,Goshen_1,Goshen,region,Northern Egypt,First mention: GEN 45:10,a60f092,https://www.openbible.info/geo/ancient/a60f092...,1,109
109,Rameses_1,Rameses,settlement,Qantir (or possibly Tell el Daba),First mention: GEN 47:11,a079b21,https://www.openbible.info/geo/ancient/a079b21...,1,110
110,Shiloh_1,Shiloh,settlement,Khirbet Seilun,First mention: GEN 49:10,aa4680a,https://www.openbible.info/geo/ancient/aa4680a...,1,111
111,Atad_1,Atad,settlement,near the mouth of the Jordan River,First mention: GEN 50:10,a2f186d,https://www.openbible.info/geo/ancient/a2f186d...,1,112
112,Pithom_1,Pithom,settlement,possibly Tell er Retaba or Tell el Maskhuta,First and only mention: EXO 1:11,a3870fe,https://www.openbible.info/geo/ancient/a3870fe...,1,113
113,Midian_1,Midian,region,East of the Gulf of Aqaba,First mention: EXO 2:15,acc6d8e,https://www.openbible.info/geo/ancient/acc6d8e...,1,114
114,Horeb_1,Horeb,mountain,Jabal al Lawz,First mention: EXO 3:1,a9bb03e,https://www.openbible.info/geo/ancient/a9bb03e...,1,115
115,Red Sea_1,Red Sea,body of water,Gulf of Aqaba,First mention: EXO 10:19,a3ba698,https://www.openbible.info/geo/ancient/a3ba698...,1,116
116,Succoth_2,Succoth,settlement,possibly Tell el Maskhuta or Jabal Maryam,First mention: EXO 12:37,aa28709,https://www.openbible.info/geo/ancient/aa28709...,2,117
117,Etham_1,Etham,settlement,possibly Ismailia,First mention: EXO 13:20,a27d0e0,https://www.openbible.info/geo/ancient/a27d0e0...,1,118


In [ ]:
OT_location_names = df_locations['place_name'].tolist()

Found this NER based BERT model that is specifically trained on finding locations by using the context of a phrase. Some limitations are that the model might not work as well on ancient location names.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")
model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")

nlp = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


The following are some keywords that we are going to use to identify the Chosen People, so that we can filter the verses that contain these key words.

In [ ]:
CHOSEN_TERMS = {'abram', 'abraham', 'hebrews', 'israelites', 'jews', 'judeans', 'people of israel'}
PATTERN = r'\b(' + '|'.join(CHOSEN_TERMS) + r')\b'

def contains_chosen_people(text: str) -> bool:
    if not isinstance(text, str):
        return False
    return bool(re.search(PATTERN, text.lower()))

In [ ]:
OT_df = OT_df[OT_df['t'].apply(contains_chosen_people)].copy()


In [ ]:
test_text = "The Jews went to Judea."
locations = nlp(test_text)
print(locations)

[{'entity_group': 'MISC', 'score': np.float32(0.9993061), 'word': 'Jews', 'start': 4, 'end': 8}, {'entity_group': 'LOC', 'score': np.float32(0.9994836), 'word': 'Judea', 'start': 17, 'end': 22}]


In [ ]:
def extract_locations(text):
    ner_results = nlp(text)
    location_tokens = [res['word'] for res in ner_results if res['entity_group'] in ["LOC"]]
    return location_tokens

locations = extract_locations(test_text)
print(locations)


['Judea']


In [ ]:
OT_df['found_locations'] = OT_df['t'].apply(extract_locations)

Now we have filtered by key words for Israelites as well as extracted locations from those filtered verses.

In [ ]:
OT_df.head(20)

,id,b,c,v,t,found_locations
292,1011026,1,11,26,"And Terah lived seventy years, and begat Abram...",[]
293,1011027,1,11,27,Now these are the generations of Terah: Terah ...,[]
295,1011029,1,11,29,And Abram and Nahor took them wives: the name ...,[]
297,1011031,1,11,31,"And Terah took Abram his son, and Lot the son ...","[Ur, Chaldees, Canaan, Haran]"
299,1012001,1,12,1,"Now the LORD had said unto Abram, Get thee out...",[]
302,1012004,1,12,4,"So Abram departed, as the LORD had spoken unto...",[Haran]
303,1012005,1,12,5,"And Abram took Sarai his wife, and Lot his bro...","[Haran, Canaan, Canaan]"
304,1012006,1,12,6,And Abram passed through the land unto the pla...,"[Sichem, Moreh]"
305,1012007,1,12,7,"And the LORD appeared unto Abram, and said, Un...",[]
307,1012009,1,12,9,"And Abram journeyed, going on still toward the...",[]


Now we are going to load another model that will help us filter the verses more by verses that contain travelling. We will use lemmas for the keywords. This will be very useful especially since we are using old English because it will use the root of each word.

In [ ]:
travel_nlp = spacy.load("en_core_web_sm")

In [ ]:
TRAVEL_LEMMAS = {
    "travel", "journey", "go", "depart", "move", "come", "leave", "cross", "return", "set", "went" "set out", "depart", "escape"
}


In [ ]:
def contains_travel_lemma(text: str) -> bool:
    if not isinstance(text, str):
        return False
    doc = travel_nlp(text) # Corrected to use the spacy model
    for token in doc:
        if token.pos_ == "VERB" and token.lemma_.lower() in TRAVEL_LEMMAS:
            return True
    return False

In [ ]:
df_travel = OT_df[df['t'].apply(
    lambda x: contains_chosen_people(x) and contains_travel_lemma(x)
)]


/tmp/ipython-input-168794251.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_travel = OT_df[df['t'].apply(


In [ ]:
df_travel.head(-5)

,id,b,c,v,t,found_locations
297,1011031,1,11,31,"And Terah took Abram his son, and Lot the son ...","[Ur, Chaldees, Canaan, Haran]"
302,1012004,1,12,4,"So Abram departed, as the LORD had spoken unto...",[Haran]
303,1012005,1,12,5,"And Abram took Sarai his wife, and Lot his bro...","[Haran, Canaan, Canaan]"
307,1012009,1,12,9,"And Abram journeyed, going on still toward the...",[]
308,1012010,1,12,10,And there was a famine in the land: and Abram ...,[Egypt]
...,...,...,...,...,...,...
19801,24033026,24,33,26,Then will I cast away the seed of Jacob and Da...,[]
19952,24040011,24,40,11,"Likewise when all the Jews that were in Moab, ...","[Moab, Edom, Babylon, Judah, ##h, Ahikam]"
19953,24040012,24,40,12,Even all the Jews returned out of all places w...,"[Judah, Gedaliah, Mizpah]"
19956,24040015,24,40,15,Then Johanan the son of Kareah spake to Gedali...,"[##eda, ##h, Mizpah, Judah]"


In [ ]:
df_travel_exploded = df_travel.explode("found_locations")
df_travel_exploded = df_travel_exploded.rename(columns={'found_locations': 'place_name'})
df_travel_exploded.head(-20)

,id,b,c,v,t,place_name
297,1011031,1,11,31,"And Terah took Abram his son, and Lot the son ...",Ur
297,1011031,1,11,31,"And Terah took Abram his son, and Lot the son ...",Chaldees
297,1011031,1,11,31,"And Terah took Abram his son, and Lot the son ...",Canaan
297,1011031,1,11,31,"And Terah took Abram his son, and Lot the son ...",Haran
302,1012004,1,12,4,"So Abram departed, as the LORD had spoken unto...",Haran
...,...,...,...,...,...,...
19952,24040011,24,40,11,"Likewise when all the Jews that were in Moab, ...",Babylon
19952,24040011,24,40,11,"Likewise when all the Jews that were in Moab, ...",Judah
19952,24040011,24,40,11,"Likewise when all the Jews that were in Moab, ...",##h
19952,24040011,24,40,11,"Likewise when all the Jews that were in Moab, ...",Ahikam


Now we map the locations we found to the corresponding modern locations.

In [ ]:
merged_df = df_travel_exploded.merge(
    df_locations[['place_name', 'modern_equivalent']],
    left_on="place_name", # Corrected from "place_names" to "place_name"
    right_on="place_name",
    how="left"
)

merged_df = merged_df.dropna(subset=["place_name"])

display(merged_df.head(10))

,id,b,c,v,t,place_name,modern_equivalent
0,1011031,1,11,31,"And Terah took Abram his son, and Lot the son ...",Ur,Tell el Muqayyar
1,1011031,1,11,31,"And Terah took Abram his son, and Lot the son ...",Chaldees,NaN
2,1011031,1,11,31,"And Terah took Abram his son, and Lot the son ...",Canaan,coastal region of northern Israel into Lebanon
3,1011031,1,11,31,"And Terah took Abram his son, and Lot the son ...",Haran,Harran
4,1012004,1,12,4,"So Abram departed, as the LORD had spoken unto...",Haran,Harran
5,1012005,1,12,5,"And Abram took Sarai his wife, and Lot his bro...",Haran,Harran
6,1012005,1,12,5,"And Abram took Sarai his wife, and Lot his bro...",Canaan,coastal region of northern Israel into Lebanon
7,1012005,1,12,5,"And Abram took Sarai his wife, and Lot his bro...",Canaan,coastal region of northern Israel into Lebanon
9,1012010,1,12,10,And there was a famine in the land: and Abram ...,Egypt,Egypt
10,1012014,1,12,14,"And it came to pass, that, when Abram was come...",Egypt,Egypt


In [ ]:
bible_coord = pd.read_csv(
    "https://www.openbible.info/geo/data/merged.txt",
    sep="\t",
    comment="#",       # ignore comment lines
    na_values=["?", ""],  # treat ? and empty as NaN
    keep_default_na=True
)

bible_coord.head()

,Abana,Unnamed: 1,33.545097,36.224661,2 Kgs 5:12,Now Barada River
0,Abarim,Mount Nebo,31.76135780447535,35.74614824478083,"Num 27:12, Num 33:47, Num 33:48, Deut 32:49, J...",NaN
1,Abdon,NaN,33.047808,35.146201,"Josh 21:30, 1 Chr 6:74",NaN
2,Abel,Abel-beth-maacah,33.26852614315892,35.57804682248786,"2 Sam 20:14, 2 Sam 20:15, 2 Sam 20:18",NaN
3,Abel-beth-maacah,NaN,33.26852614315892,35.57804682248786,"1 Kgs 15:20, 2 Kgs 15:29",NaN
4,Abel-keramim,NaN,~32.02425,~35.82352,Judg 11:33,NaN


In [ ]:
bible_coord = bible_coord.rename(columns={'Abana': 'place', '33.545097': 'Lat', '36.224661': 'Long'})
bible_coord = bible_coord.drop(columns=['Unnamed: 1', '2 Kgs 5:12', 'Now Barada River'], errors='ignore')

In [ ]:
bible_coord.head()

,place,Lat,Long
0,Abarim,31.76135780447535,35.74614824478083
1,Abdon,33.047808,35.146201
2,Abel,33.26852614315892,35.57804682248786
3,Abel-beth-maacah,33.26852614315892,35.57804682248786
4,Abel-keramim,~32.02425,~35.82352


In [ ]:
def clean_and_convert_coord(coord_series):
    # Replace '~' character from the beginning of the string
    cleaned_series = coord_series.astype(str).str.replace('^~', '', regex=True)
    # Convert to numeric, coercing errors to NaN
    return pd.to_numeric(cleaned_series, errors='coerce')

# Apply the cleaning and conversion to 'Lat' and 'Long' columns
bible_coord['Lat'] = clean_and_convert_coord(bible_coord['Lat'])
bible_coord['Long'] = clean_and_convert_coord(bible_coord['Long'])

# Display the cleaned DataFrame head and info to verify changes
bible_coord.head()

,place,Lat,Long
0,Abarim,31.761358,35.746148
1,Abdon,33.047808,35.146201
2,Abel,33.268526,35.578047
3,Abel-beth-maacah,33.268526,35.578047
4,Abel-keramim,32.024250,35.823520


In [ ]:
merged_df_2 = merged_df.merge(
    bible_coord[['place', 'Lat', 'Long']],
    left_on="place_name",
    right_on="place", # Changed from "place_name" to "place"
    how="left"
)

merged_df_2.head(-10)

,id,b,c,v,t,place_name,modern_equivalent,place,Lat,Long
0,1011031,1,11,31,"And Terah took Abram his son, and Lot the son ...",Ur,Tell el Muqayyar,Ur,30.962052,46.103742
1,1011031,1,11,31,"And Terah took Abram his son, and Lot the son ...",Chaldees,NaN,NaN,NaN,NaN
2,1011031,1,11,31,"And Terah took Abram his son, and Lot the son ...",Canaan,coastal region of northern Israel into Lebanon,Canaan,NaN,NaN
3,1011031,1,11,31,"And Terah took Abram his son, and Lot the son ...",Haran,Harran,Haran,36.863864,39.032197
4,1012004,1,12,4,"So Abram departed, as the LORD had spoken unto...",Haran,Harran,Haran,36.863864,39.032197
...,...,...,...,...,...,...,...,...,...,...
102,24040015,24,40,15,Then Johanan the son of Kareah spake to Gedali...,Mizpah,NaN,NaN,NaN,NaN
103,24040015,24,40,15,Then Johanan the son of Kareah spake to Gedali...,Judah,NaN,NaN,NaN,NaN
104,24044001,24,44,1,The word that came to Jeremiah concerning all ...,Egypt,Egypt,Egypt,NaN,NaN
105,24044001,24,44,1,The word that came to Jeremiah concerning all ...,Migdol,NaN,Migdol,NaN,NaN


In [ ]:
merged_df_2 = merged_df_2.dropna(subset=['modern_equivalent'])
merged_df_2.head()

,id,b,c,v,t,place_name,modern_equivalent,place,Lat,Long
0,1011031,1,11,31,"And Terah took Abram his son, and Lot the son ...",Ur,Tell el Muqayyar,Ur,30.962052,46.103742
2,1011031,1,11,31,"And Terah took Abram his son, and Lot the son ...",Canaan,coastal region of northern Israel into Lebanon,Canaan,NaN,NaN
3,1011031,1,11,31,"And Terah took Abram his son, and Lot the son ...",Haran,Harran,Haran,36.863864,39.032197
4,1012004,1,12,4,"So Abram departed, as the LORD had spoken unto...",Haran,Harran,Haran,36.863864,39.032197
5,1012005,1,12,5,"And Abram took Sarai his wife, and Lot his bro...",Haran,Harran,Haran,36.863864,39.032197


In [ ]:
merged_df_2.to_csv('file1.csv')

In [ ]:
merged_df_2['place'].isna().sum()

np.int64(3)

In [ ]:
# 33.60147585041282, 35.762108139105
# 30.648333, 34.422222
col = 'place'
left_col = merged_df_2.columns[merged_df_2.columns.get_loc(col) - 1]
print(merged_df_2.loc[merged_df_2[col].isna(), left_col])

21    Ain el Qudeirat
47            Et Tell
48            Et Tell
Name: modern_equivalent, dtype: object


In [ ]:
merged_df_2.loc[4, 'Lat'] = 30.648333
merged_df_2.loc[4, 'Long'] = 34.422222

merged_df_2.loc[32, 'Lat'] = 33.60147585041282
merged_df_2.loc[32, 'Long'] = 35.762108139105

merged_df_2.loc[33, 'Lat'] = 33.60147585041282
merged_df_2.loc[33, 'Long'] = 35.762108139105

In [ ]:
mask = merged_df_2["modern_equivalent"].str.strip().eq("Egypt")

merged_df_2.loc[mask, "Lat"] = 30.108086
merged_df_2.loc[mask, "Long"] = 31.338220

mask2 = merged_df_2["modern_equivalent"].str.strip().eq("Et Tell")

merged_df_2.loc[mask2, "Lat"] = 33.60147585041282
merged_df_2.loc[mask2, "Long"] = 35.762108139105

mask3 = merged_df_2["modern_equivalent"].str.strip().eq("Ain el Qudeirat")

merged_df_2.loc[mask3, "Lat"] = 30.648333
merged_df_2.loc[mask3, "Long"] = 34.422222

mask4 = merged_df_2["place"].str.strip().eq("Canaan")

merged_df_2.loc[mask4, "Lat"] = 31.69352950368834
merged_df_2.loc[mask4, "Long"] = 34.84388288576839



In [ ]:
merged_df_2.head()

,id,b,c,v,t,place_name,modern_equivalent,place,Lat,Long
0,1011031.0,1.0,11.0,31.0,"And Terah took Abram his son, and Lot the son ...",Ur,Tell el Muqayyar,Ur,30.962052,46.103742
2,1011031.0,1.0,11.0,31.0,"And Terah took Abram his son, and Lot the son ...",Canaan,coastal region of northern Israel into Lebanon,Canaan,31.693530,34.843883
3,1011031.0,1.0,11.0,31.0,"And Terah took Abram his son, and Lot the son ...",Haran,Harran,Haran,36.863864,39.032197
4,1012004.0,1.0,12.0,4.0,"So Abram departed, as the LORD had spoken unto...",Haran,Harran,Haran,30.648333,34.422222
5,1012005.0,1.0,12.0,5.0,"And Abram took Sarai his wife, and Lot his bro...",Haran,Harran,Haran,36.863864,39.032197


Now we can use our cleaned dataframe to make a chronological timeline of where the Israelites travelled (modern equivalent). P.S. The whole timeline may not be visible in the cell output.

In [ ]:
places = [str(x) for x in merged_df_2['modern_equivalent'].tolist()]
filtered = [places[0]] + [places[i] for i in range(1, len(places)) if places[i] != places[i-1]]

# Join with arrows
result = " → ".join(filtered)
print(result)

Tell el Muqayyar → coastal region of northern Israel into Lebanon → Harran → coastal region of northern Israel into Lebanon → Egypt → Ramat el Khalil → Damascus → in the valley south of the Lisan peninsula → Ain el Qudeirat → Ar Ruwaysat → Tel Haror → Tel Beer Sheva (near Bir es Seba) → Tel Rumeida → coastal region of northern Israel into Lebanon → Tel Haror → Ramat el Khalil → Tel Rumeida → Egypt → Et Tell → Khirbet Seilun → Jordan river → hill country west-northwest of modern Amman and loosely associated with the Jabbok River → Paran → partly Turkey and partly Iraq → Jerusalem → Area east of the Dead Sea → A region between the Dead Sea and the Gulf of Aqaba → Egypt → nan


In [ ]:
print(len(filtered))

29


Cleaned it up because Egypt in that context was not properly identified.

In [ ]:
final_df = merged_df_2.iloc[:-2]
places = [str(x) for x in final_df['modern_equivalent'].tolist()]
filtered = [places[0]] + [places[i] for i in range(1, len(places)) if places[i] != places[i-1]]

# Join with arrows
result = " → ".join(filtered)
print(result)

Tell el Muqayyar → coastal region of northern Israel into Lebanon → Harran → coastal region of northern Israel into Lebanon → Egypt → Ramat el Khalil → Damascus → in the valley south of the Lisan peninsula → Ain el Qudeirat → Ar Ruwaysat → Tel Haror → Tel Beer Sheva (near Bir es Seba) → Tel Rumeida → coastal region of northern Israel into Lebanon → Tel Haror → Ramat el Khalil → Tel Rumeida → Egypt → Et Tell → Khirbet Seilun → Jordan river → hill country west-northwest of modern Amman and loosely associated with the Jabbok River → Paran → partly Turkey and partly Iraq → Jerusalem → Area east of the Dead Sea → A region between the Dead Sea and the Gulf of Aqaba → Egypt


So that we can see the whole output better.

In [ ]:
places = [str(x) for x in final_df['modern_equivalent'].tolist()]
filtered = [places[0]] + [places[i] for i in range(1, len(places)) if places[i] != places[i-1]]
result = "\n↓\n".join(filtered)
print(result)

Tell el Muqayyar
↓
coastal region of northern Israel into Lebanon
↓
Harran
↓
coastal region of northern Israel into Lebanon
↓
Egypt
↓
Ramat el Khalil
↓
Damascus
↓
in the valley south of the Lisan peninsula
↓
Ain el Qudeirat
↓
Ar Ruwaysat
↓
Tel Haror
↓
Tel Beer Sheva (near Bir es Seba)
↓
Tel Rumeida
↓
coastal region of northern Israel into Lebanon
↓
Tel Haror
↓
Ramat el Khalil
↓
Tel Rumeida
↓
Egypt
↓
Et Tell
↓
Khirbet Seilun
↓
Jordan river
↓
hill country west-northwest of modern Amman and loosely associated with the Jabbok River
↓
Paran
↓
partly Turkey and partly Iraq
↓
Jerusalem
↓
Area east of the Dead Sea
↓
A region between the Dead Sea and the Gulf of Aqaba
↓
Egypt


Now we can plot the locations we have found.

In [ ]:
df_to_plot = final_df.copy()

fig = px.scatter_mapbox(
    df_to_plot,
    lat="Lat",
    lon="Long",
    hover_name="place_name",
    hover_data=["modern_equivalent"],
    color_discrete_sequence=px.colors.qualitative.Plotly,
    zoom=4,
    height=650,
)

fig.update_layout(
    mapbox_style="carto-positron",
    title="<b>Travel Locations Mentioned in the Hebrew Bible</b>",
    title_x=0.5,
    margin={"r":0, "t":50, "l":0, "b":0},
    mapbox=dict(
        pitch=10,
        bearing=0
    )
)

fig.show()

This has been an expirament. More on the model's limitations in the writeup.